# A Gallery of Common Distributions

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 03.05 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/06_common_distributions_gallery.ipynb)

---

## 🎯 Learning Objective

Tour the distributions that appear constantly in ML — Gaussian, Bernoulli, Categorical, Poisson, Exponential, Beta, Dirichlet, Gamma — and learn when each one is the natural choice.

---

## 📐 Theory

You now have the full grammar of probability (`03.01`-`03.05`). This notebook is a field guide
to the specific distributions that recur constantly in ML — what each one models, its
parameters, and its moments in terms of those parameters.

### Bernoulli and Categorical: the building blocks of classification

**Bernoulli($p$)** is a single coin flip: $X\in\{0,1\}$, $P(X=1)=p$, $\mathbb{E}[X]=p$,
$\text{Var}(X)=p(1-p)$. **Categorical($p_1,\ldots,p_K$)** generalizes it to $K$ outcomes each
with its own probability, $\sum_k p_k=1$ — exactly the distribution a softmax layer produces
over classes or vocabulary tokens.

### Binomial and Poisson: counting things

**Binomial($n,p$)** counts successes in $n$ independent Bernoulli($p$) trials: $P(X=k) =
\binom{n}{k}p^k(1-p)^{n-k}$, $\mathbb{E}[X]=np$, $\text{Var}(X)=np(1-p)$ — the sum of $n$ iid
Bernoullis, by the linearity of expectation from `03.03`. **Poisson($\lambda$)** counts events
in a fixed interval when events occur independently at a constant average rate $\lambda$:
$P(X=k) = \lambda^k e^{-\lambda}/k!$, with the unusual property $\mathbb{E}[X]=\text{Var}(X)=
\lambda$. Poisson is literally the limit of Binomial($n,p$) as $n\to\infty$, $p\to0$, with
$np=\lambda$ fixed — many rare independent chances collapsing into one rate.

### Gaussian: the default continuous distribution

**Gaussian/Normal($\mu,\sigma^2$)**: $f(x) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$,
$\mathbb{E}[X]=\mu$, $\text{Var}(X)=\sigma^2$. Its ubiquity isn't arbitrary: the **Central Limit
Theorem** says the sum of many independent, identically distributed random variables
approaches a Gaussian regardless of their own distribution — noise from many small independent
sources tends to look Gaussian, which is exactly why it's the default noise/prior assumption
across ML.

### Exponential and Gamma: waiting times

**Exponential($\lambda$)** models the waiting time until a Poisson-rate event occurs:
$f(x)=\lambda e^{-\lambda x}$ for $x\ge0$, $\mathbb{E}[X]=1/\lambda$. It's the unique continuous
distribution with the **memoryless property** $P(X>s+t\mid X>s)=P(X>t)$ — how long you've
already waited tells you nothing about how much longer you'll wait. **Gamma($k,\lambda$)**
generalizes it to the waiting time until the $k$-th event, and is the conjugate prior for a
Poisson rate or for the *precision* (inverse variance) of a Gaussian.

### Beta and Dirichlet: distributions over probabilities

**Beta($\alpha,\beta$)** is a distribution over $[0,1]$ — perfect for representing uncertainty
about a probability itself (a coin's bias, in `03.05`'s example), with $\mathbb{E}[X] =
\alpha/(\alpha+\beta)$. It is the conjugate prior to the Bernoulli/Binomial likelihood.
**Dirichlet($\alpha_1,\ldots,\alpha_K$)** is Beta's multivariate generalization: a distribution
over the probability simplex $\{(p_1,\ldots,p_K): p_k\ge0,\sum_k p_k=1\}$ — a distribution over
*entire categorical distributions*. It is conjugate to the Categorical likelihood, making it the
natural prior whenever a model's parameters are themselves a probability vector (topic
proportions in topic models, class priors, attention-weight priors).

### Choosing a distribution: matching support to structure

The right distribution is the one whose **support** (the set of values it can take) and
built-in assumptions match your data or parameter's actual constraints: binary outcomes →
Bernoulli; one-of-$K$ choices → Categorical; non-negative counts → Poisson; non-negative
continuous, possibly skewed → Exponential/Gamma; a bounded probability → Beta; a vector that
must sum to 1 → Dirichlet; everything else, by default and by the Central Limit Theorem →
Gaussian. The table built below turns this into a concrete lookup.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A gallery of the distributions above, each plotted with a few different parameter settings, so
their shapes and how their parameters control them are visible side by side.

In [ ]:
# A gallery: each distribution plotted at a few parameter settings, using the SAME
# .pdf()/.pmf() pattern established in 03.02 -- the API is identical no matter the distribution.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

k_vals = np.arange(0, 11)
for n, p in [(10, 0.3), (10, 0.5), (10, 0.7)]:
    axes[0, 0].plot(k_vals, stats.binom(n, p).pmf(k_vals), "o-", label=f"n={n}, p={p}", ms=4)
axes[0, 0].set_title("Binomial(n, p)"); axes[0, 0].legend(fontsize=8); axes[0, 0].set_xlabel("k")

for lam in [1, 4, 9]:
    axes[0, 1].plot(k_vals, stats.poisson(lam).pmf(k_vals), "o-", label=f"lambda={lam}", ms=4)
axes[0, 1].set_title("Poisson(lambda)"); axes[0, 1].legend(fontsize=8); axes[0, 1].set_xlabel("k")

x_gauss = np.linspace(-6, 6, 300)
for mu, sigma in [(0, 1), (0, 2), (-2, 0.7)]:
    axes[0, 2].plot(x_gauss, stats.norm(mu, sigma).pdf(x_gauss), label=f"mu={mu}, sigma={sigma}")
axes[0, 2].set_title("Gaussian(mu, sigma)"); axes[0, 2].legend(fontsize=8); axes[0, 2].set_xlabel("x")

x_pos = np.linspace(0, 5, 300)
for lam in [0.5, 1, 2]:
    axes[1, 0].plot(x_pos, stats.expon(scale=1 / lam).pdf(x_pos), label=f"lambda={lam}")
axes[1, 0].set_title("Exponential(lambda)"); axes[1, 0].legend(fontsize=8); axes[1, 0].set_xlabel("x")

for k_shape, theta_scale in [(1, 1), (2, 1), (5, 1)]:
    axes[1, 1].plot(x_pos, stats.gamma(k_shape, scale=theta_scale).pdf(x_pos), label=f"k={k_shape}, theta={theta_scale}")
axes[1, 1].set_title("Gamma(k, theta) [k=1 -> Exponential]"); axes[1, 1].legend(fontsize=8); axes[1, 1].set_xlabel("x")

x_unit = np.linspace(0.001, 0.999, 300)
for a, b in [(0.5, 0.5), (2, 2), (2, 5)]:
    axes[1, 2].plot(x_unit, stats.beta(a, b).pdf(x_unit), label=f"alpha={a}, beta={b}")
axes[1, 2].set_title("Beta(alpha, beta) -- lives on [0,1]"); axes[1, 2].legend(fontsize=8); axes[1, 2].set_xlabel("x")

plt.tight_layout()
plt.show()

print("Binomial(10,0.5) mean:", stats.binom(10, 0.5).mean(), " (expect n*p=5.0)")
print("Poisson(4) mean == variance:", stats.poisson(4).mean(), "==", stats.poisson(4).var(), " (defining property)")
print("Gamma(1, scale=2) matches Exponential(scale=2)? ",
      np.allclose(stats.gamma(1, scale=2).pdf(x_pos), stats.expon(scale=2).pdf(x_pos)))

## 🐍 Implementation from Scratch

We implement the Categorical and Dirichlet distributions from scratch (sampling and density),
verify the Binomial-as-limit-of-Poisson connection numerically, and cross-check everything
against `scipy.stats`.

In [ ]:
rng = np.random.default_rng(0)

# Categorical sampling from scratch: inverse-transform sampling (03.02) applied to a PMF over
# a finite unordered set of outcomes, via the CDF of the category INDEX.
def categorical_sample_from_scratch(probs, size, rng):
    cdf = np.cumsum(probs)
    u = rng.uniform(0, 1, size=size)
    return np.searchsorted(cdf, u)  # first index where cdf exceeds u -- inverse transform

category_probs = np.array([0.1, 0.2, 0.4, 0.3])
samples = categorical_sample_from_scratch(category_probs, size=50_000, rng=rng)
empirical_freqs = np.bincount(samples, minlength=4) / len(samples)
print("Categorical from-scratch empirical frequencies:", np.round(empirical_freqs, 3))
print("True probabilities:                             ", category_probs)

# Dirichlet PDF from scratch, straight from its definition, checked against scipy.stats.dirichlet
def dirichlet_pdf_from_scratch(x, alpha):
    # f(x) = [1/B(alpha)] * prod_k x_k^(alpha_k - 1), B(alpha) = prod Gamma(alpha_k) / Gamma(sum alpha_k)
    log_beta_norm = np.sum(gammaln(alpha)) - gammaln(np.sum(alpha))
    log_pdf = np.sum((alpha - 1) * np.log(x)) - log_beta_norm
    return np.exp(log_pdf)

alpha_vec = np.array([2.0, 3.0, 4.0])
test_point = np.array([0.2, 0.3, 0.5])
ours = dirichlet_pdf_from_scratch(test_point, alpha_vec)
scipy_answer = stats.dirichlet(alpha_vec).pdf(test_point)
print(f"\nDirichlet PDF at {test_point}: ours={ours:.6f}, scipy={scipy_answer:.6f}, "
      f"diff={abs(ours - scipy_answer):.2e}")

# A Dirichlet sample IS itself a valid categorical distribution -- sample one, check it sums to 1
one_dirichlet_draw = stats.dirichlet(alpha_vec).rvs(random_state=0)[0]
print(f"One Dirichlet({alpha_vec}) draw: {np.round(one_dirichlet_draw, 4)} (sums to {one_dirichlet_draw.sum():.6f})")

# Binomial -> Poisson limit: fix lambda = n*p, shrink p while growing n, and watch the
# Binomial PMF converge pointwise to the Poisson(lambda) PMF
lam_fixed = 3.0
k_vals = np.arange(0, 10)
poisson_pmf = stats.poisson(lam_fixed).pmf(k_vals)
print(f"\nPoisson(lambda={lam_fixed}) PMF:              {np.round(poisson_pmf, 5)}")
for n in [30, 300, 3000]:
    p = lam_fixed / n
    binom_pmf = stats.binom(n, p).pmf(k_vals)
    max_diff = np.max(np.abs(binom_pmf - poisson_pmf))
    print(f"Binomial(n={n:4d}, p={p:.5f}) PMF:  max diff from Poisson = {max_diff:.6f}")
print("(max difference shrinks as n grows -- confirms Poisson is the n->inf, p->0, np=lambda limit)")

## 🤖 Why This Matters for AI

Every classification model's output layer is a **Categorical** distribution (softmax); every
topic model or Bayesian treatment of class proportions reaches for a **Dirichlet** prior because
it's conjugate to Categorical, turning posterior updates into simple parameter addition rather
than an intractable integral. Below, we run that update explicitly: start from a Dirichlet prior
over a language model's next-token distribution, observe token counts, and get a closed-form
Dirichlet posterior — the categorical/Dirichlet analogue of `03.05`'s Beta-Bernoulli example.

---

In [ ]:
# Dirichlet-Categorical conjugacy: a Bayesian view of "estimating a language model's next-token
# distribution" -- start with a prior belief, observe token counts, get a closed-form posterior.
vocab = ["the", "a", "cat", "dog", "runs"]
V = len(vocab)

# A weak, symmetric Dirichlet prior: mild belief that all tokens are roughly equally likely
prior_alpha = np.ones(V) * 2.0

# Observed counts of each token in a small corpus sample
observed_counts = np.array([40, 25, 15, 12, 8])
print(f"Vocabulary: {vocab}")
print(f"Observed counts: {observed_counts}  (total {observed_counts.sum()} tokens seen)")

# Dirichlet-Categorical conjugacy: posterior alpha = prior alpha + observed counts. Closed form,
# exactly like the Beta-Bernoulli update in 03.05, just generalized from 2 categories to V.
posterior_alpha = prior_alpha + observed_counts
posterior_mean_dist = posterior_alpha / posterior_alpha.sum()  # E[p_k] for a Dirichlet(alpha) is alpha_k/sum(alpha)

# Compare to the plain MLE estimate (normalized counts, no prior at all -- 03.07's approach)
mle_dist = observed_counts / observed_counts.sum()

print("\nToken   |  MLE estimate  |  Bayesian posterior mean")
for tok, mle_p, post_p in zip(vocab, mle_dist, posterior_mean_dist):
    print(f"{tok:7s} |     {mle_p:.4f}     |          {post_p:.4f}")

# The prior matters most when data is SCARCE -- rerun with far fewer observed tokens
sparse_counts = np.array([2, 1, 0, 0, 0])   # "dog" and "runs" never observed yet
posterior_alpha_sparse = prior_alpha + sparse_counts
posterior_mean_sparse = posterior_alpha_sparse / posterior_alpha_sparse.sum()
mle_sparse = sparse_counts / sparse_counts.sum()

print(f"\nWith only {sparse_counts.sum()} tokens observed (two tokens never seen):")
print(f"  MLE estimate           : {np.round(mle_sparse, 4)}  <- assigns EXACTLY ZERO probability to unseen tokens")
print(f"  Bayesian posterior mean: {np.round(posterior_mean_sparse, 4)}  <- prior keeps unseen tokens' probability > 0")
print("This is exactly why Bayesian/Dirichlet smoothing (a direct ancestor of Laplace/add-k")
print("smoothing) matters for language models: it avoids assigning zero probability -- and")
print("hence infinite surprise/cross-entropy -- to a token never observed during training.")

## 🏋️ Exercises

### Warm-up
1. Using `scipy.stats`, compute the mean and variance of `Gamma(k=3, theta=2)` two ways: (a)
   via `.mean()`/`.var()`, and (b) via the formulas $\mathbb{E}[X]=k\theta$,
   $\text{Var}(X)=k\theta^2$. Confirm they match.

### Practice
2. Verify the Beta-Binomial connection: for a Beta($\alpha,\beta$) prior updated with $k$
   successes out of $n$ Bernoulli trials, the posterior is Beta($\alpha+k,\beta+n-k$) (used in
   `03.05`). Pick concrete numbers, compute the posterior mean in closed form, and confirm it
   matches `scipy.stats.beta(...).mean()`.

### Challenge
3. In the Dirichlet-Categorical smoothing cell, sweep the prior strength by scaling
   `prior_alpha` by factors `[0.1, 1, 10, 100]` (keeping it symmetric) while holding
   `sparse_counts` fixed. Plot the posterior mean probability assigned to a never-observed
   token ("dog") against the prior strength. Explain the trend, and connect it to the
   bias-variance tradeoff: why might a language model want a small but nonzero prior strength
   rather than the largest one that maximizes robustness to unseen tokens?

---

## 📚 Further Reading
- Murphy, *Probabilistic Machine Learning: An Introduction*, Ch. 2.2–2.3 (Common probability distributions)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 2 (Probability Distributions)
- `scipy.stats` [continuous distributions reference](https://docs.scipy.org/doc/scipy/reference/stats.html#continuous-distributions)

---

*Next notebook: [Maximum Likelihood Estimation](07_maximum_likelihood_estimation.ipynb)*